### Notebook structure
| # | Section |
|---|---------|
| 1 | Verify dataset path |
| 2 | Install dependencies |
| 3 | 7-channel preprocessing (ELA + DCT) — Novelty N1 |
| 4 | Cache all 7-ch tensors to disk |
| 5 | FaceDataset class |
| 6 | Platform Non-IID partitioning (N6) |
| 7 | FedDeepGuard-ViT model + FA-RGA gate (N1) |
| 8 | Local training function |
| 9 | Attack injection — A1 (GAN poisoning) + A2 (freq backdoor) — N2/N3 |
| 10 | Defence — DCT gradient fidelity scoring (N4) |
| 11 | Blockchain audit logger (N5) |
| 12 | Baseline aggregation methods |
| 13 | Full 50-round FL training loop |
| 14 | Evaluation — 9 metrics |
| 15 | Run all experiments → save to Excel |


# Verify dataset path and Train, Test, Valid split

In [1]:
from pathlib import Path
import os

DATA_ROOT = Path('/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake')

assert DATA_ROOT.exists(), f"Still not found at {DATA_ROOT}"

for split in ['train', 'valid', 'test']:
    real_count = len(list((DATA_ROOT / split / 'real').glob('*.jpg')))
    fake_count = len(list((DATA_ROOT / split / 'fake').glob('*.jpg')))
    print(f"{split:<6} → real: {real_count:>6,}  fake: {fake_count:>6,}  total: {real_count+fake_count:>7,}")

train  → real: 50,000  fake: 50,000  total: 100,000
valid  → real: 10,000  fake: 10,000  total:  20,000
test   → real: 10,000  fake: 10,000  total:  20,000


# Install dependencies

In [2]:
!pip install -q timm einops flwr[simulation] scikit-learn opacus


ERROR: Could not find a version that satisfies the requirement opacus (from versions: none)
ERROR: No matching distribution found for opacus


# 7-channel preprocessing

Each image → 7 channels:
- Channels 0–2: RGB (visual face appearance)
- Channel 3: ELA — Error Level Analysis (JPEG re-compression artifact map)
- Channel 4: DCT low-frequency (0–8 coefficients)
- Channel 5: DCT mid-frequency (8–32 coefficients) ← **highest GAN signal**
- Channel 6: DCT high-frequency (32–64 coefficients)


In [3]:
 """
    Error Level Analysis: re-save at quality=90 and measure deviation.
    Real photos → uniform error. GAN images → anomalous patterns.
    Returns: (256, 256) uint8 array, normalised to [0, 255].
    """


import cv2
import numpy as np
from PIL import Image
import io

#  ELA channel 
def compute_ela(img_path, quality=90):
   
    img = Image.open(img_path).convert('RGB')

    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    recompressed = Image.open(buf).convert('RGB')

    ela = np.abs(
        np.array(img).astype(np.float32) -
        np.array(recompressed).astype(np.float32)
    )
    # Average across RGB channels, normalise to 0–255
    ela = (ela.mean(axis=2) / (ela.max() + 1e-8) * 255.0).astype(np.uint8)
    return ela   # (256, 256)



 """
    Compute 3 DCT frequency-band channels from a grayscale image.
    Returns three (256, 256) float32 arrays: low, mid, high.
    """

# DCT 3 channels
def compute_dct_channels(img_path):
   
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"cv2 could not read: {img_path}")

    img = img.astype(np.float32) / 255.0   # normalise to [0, 1]

    # Full 2D DCT transform → (256, 256)
    dct = cv2.dct(img)

    # Three frequency bands by DCT coefficient index
    low_region  = np.abs(dct[:8,  :8 ])   # 8×8   coarse structure
    mid_region  = np.abs(dct[:32, :32])   # 32×32 GAN periodic patterns
    high_region = np.abs(dct[:64, :64])   # 64×64 fine texture residuals

    def to_256(arr):
        return cv2.resize(arr, (256, 256), interpolation=cv2.INTER_LINEAR)

    return to_256(low_region), to_256(mid_region), to_256(high_region)


# Quick sanity check on one real image
sample_img = str(next((DATA_ROOT / 'train' / 'real').glob('*.jpg')))
ela_test = compute_ela(sample_img)
l, m, h = compute_dct_channels(sample_img)
print(f"ELA  shape: {ela_test.shape}  dtype: {ela_test.dtype}  range: [{ela_test.min()}, {ela_test.max()}]")
print(f"DCT-low  : {l.shape}  min={l.min():.4f}  max={l.max():.4f}")
print(f"DCT-mid  : {m.shape}  min={m.min():.4f}  max={m.max():.4f}")
print(f"DCT-high : {h.shape}  min={h.min():.4f}  max={h.max():.4f}")


ELA  shape: (256, 256)  dtype: uint8  range: [0, 139]
DCT-low  : (256, 256)  min=0.0579  max=99.0939
DCT-mid  : (256, 256)  min=0.0045  max=99.0939
DCT-high : (256, 256)  min=0.0049  max=99.0939


# Cache all 7-channel tensors to disk

In [4]:
import cv2
import numpy as np
from PIL import Image
import io
import torch
from torch.utils.data import Dataset, DataLoader

def compute_7ch_tensor(img_path):
    img_path = str(img_path)
    
    # RGB
    rgb = cv2.imread(img_path)
    if rgb is None:
        return None
    rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (256, 256))
    
    # ELA
    pil = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil.save(buf, format='JPEG', quality=90)
    buf.seek(0)
    recomp = np.array(Image.open(buf))
    ela = np.abs(rgb.astype(np.float32) - recomp.astype(np.float32))
    ela = (ela.mean(axis=2) / (ela.max() + 1e-8) * 255).astype(np.uint8)
    
    # DCT
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    dct  = cv2.dct(gray)
    def to_256(a): return cv2.resize(np.abs(a), (256, 256))
    low  = to_256(dct[:8,  :8 ])
    mid  = to_256(dct[:32, :32])
    high = to_256(dct[:64, :64])
    
    arr = np.stack([
        rgb[...,0], rgb[...,1], rgb[...,2],
        ela, low, mid, high
    ], axis=0).astype(np.float32) / 255.0
    
    return torch.tensor(arr)


class FaceDataset(Dataset):
    def __init__(self, root, split='train'):
        split_dir = Path(root) / split
        self.files, self.labels = [], []
        for label, cls in enumerate(['real', 'fake']):
            for f in sorted((split_dir / cls).glob('*.jpg')):
                self.files.append(str(f))
                self.labels.append(label)
        print(f"FaceDataset [{split}]  "
              f"real={self.labels.count(0):,}  "
              f"fake={self.labels.count(1):,}")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        tensor = compute_7ch_tensor(self.files[idx])
        if tensor is None:
            # Return zeros for corrupt files (rare)
            tensor = torch.zeros(7, 256, 256)
        return tensor, self.labels[idx]


DATA_ROOT = Path('/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake')

train_dataset = FaceDataset(DATA_ROOT, split='train')
valid_dataset = FaceDataset(DATA_ROOT, split='valid')
test_dataset  = FaceDataset(DATA_ROOT, split='test')

# num_workers=4 parallelises the on-the-fly computation — fast enough
val_loader  = DataLoader(valid_dataset, batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)

# Quick check
t, y = train_dataset[0]
print(f"Tensor shape : {t.shape}")   # (7, 256, 256)
print(f"Tensor range : [{t.min():.3f}, {t.max():.3f}]")
print("Ready — no disk cache needed")

FaceDataset [train]  real=50,000  fake=50,000
FaceDataset [valid]  real=10,000  fake=10,000
FaceDataset [test]  real=10,000  fake=10,000
Tensor shape : torch.Size([7, 256, 256])
Tensor range : [0.000, 1.000]
Ready — no disk cache needed


# Platform-based Non-IID partitioning — Novelty N6

In [5]:
import numpy as np
from torch.utils.data import Subset

PLATFORMS = {
    0: 'Twitter/Instagram',  1: 'TikTok/YouTube',  2: 'Facebook/Reddit',
    3: 'Reuters/AP',         4: 'BBC/CNN',          5: 'NYT/Guardian',
    6: 'Forums/Discord',     7: 'Personal sites',   8: 'Cloud storage',
    9: 'Email/messaging'
}


def platform_noniid_split(dataset, n_clients=10, alpha=1.0, seed=42):
    """
    Dirichlet Non-IID split across n_clients.
    alpha=1.0 → moderate heterogeneity (paper setting).
    alpha=0.1 → extreme heterogeneity (stress test).
    Returns: dict {client_id: np.array of dataset indices}
    """
    rng    = np.random.default_rng(seed)
    labels = np.array(dataset.labels)

    idx_real = np.where(labels == 0)[0]
    idx_fake = np.where(labels == 1)[0]
    rng.shuffle(idx_real)
    rng.shuffle(idx_fake)

    props_real = rng.dirichlet([alpha] * n_clients)
    props_fake = rng.dirichlet([alpha] * n_clients)

    client_indices = {}
    real_ptr, fake_ptr = 0, 0

    for i in range(n_clients):
        n_real = int(props_real[i] * len(idx_real))
        n_fake = int(props_fake[i] * len(idx_fake))

        real_chunk = idx_real[real_ptr : real_ptr + n_real]
        fake_chunk = idx_fake[fake_ptr : fake_ptr + n_fake]

        client_indices[i] = np.concatenate([real_chunk, fake_chunk])
        real_ptr += n_real
        fake_ptr += n_fake

    return client_indices


def log_client_distribution(dataset, client_indices):
    """Print per-client distribution table (→ Table T2)."""
    print(f"{'Client':<7} {'Platform':<22} {'Total':>7} {'Real':>7} {'Fake':>7} {'FakeRatio':>10}")
    print('─' * 65)
    for cid, indices in client_indices.items():
        lbls   = np.array(dataset.labels)[indices]
        n_real = (lbls == 0).sum()
        n_fake = (lbls == 1).sum()
        total  = len(lbls)
        ratio  = n_fake / total if total > 0 else 0
        print(f"C{cid+1:<6} {PLATFORMS[cid]:<22} {total:>7,} {n_real:>7,} {n_fake:>7,} {ratio:>9.1%}")
    print('─' * 65)
    print(f"{'TOTAL':<7} {'':22} {sum(len(v) for v in client_indices.values()):>7,}")


# Generate and display partition (Table T2)
client_indices = platform_noniid_split(train_dataset, alpha=1.0, seed=42)
log_client_distribution(train_dataset, client_indices)


def get_client_loader(cid, batch_size=32):
    """Return a DataLoader for one client's local subset."""
    subset = Subset(train_dataset, client_indices[cid])
    return DataLoader(subset, batch_size=batch_size, shuffle=True,
                      num_workers=2, pin_memory=True)

print("\nNon-IID partitioning complete")


Client  Platform                 Total    Real    Fake  FakeRatio
─────────────────────────────────────────────────────────────────
C1      Twitter/Instagram        4,338   3,832     506     11.7%
C2      TikTok/YouTube           2,853   1,644   1,209     42.4%
C3      Facebook/Reddit          8,179   5,801   2,378     29.1%
C4      Reuters/AP              16,422  10,793   5,629     34.3%
C5      BBC/CNN                  3,303     551   2,752     83.3%
C6      NYT/Guardian            11,008   8,253   2,755     25.0%
C7      Forums/Discord          10,633   2,650   7,983     75.1%
C8      Personal sites          25,387   3,754  21,633     85.2%
C9      Cloud storage           14,265  10,179   4,086     28.6%
C10     Email/messaging          3,603   2,539   1,064     29.5%
─────────────────────────────────────────────────────────────────
TOTAL                           99,991

Non-IID partitioning complete


# FedDeepGuard-ViT model + FA-RGA gate — Novelty N1

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import copy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


class FARGAGate(nn.Module):
    """
    Frequency-Aware Region-Guided Attention gate — Novelty N1.
    Learns a scalar gate per spatial patch: [0, 1].
    High gate = patch contains strong GAN artifact signal.
    Low gate  = patch is uninformative background.
    """
    def __init__(self, embed_dim=384):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, patch_tokens):
        # patch_tokens: (B, N_patches, D)  e.g. (B, 256, 384)
        weights = self.gate(patch_tokens)          # (B, 256, 1)
        return patch_tokens * weights              # element-wise gate


class FedDeepGuardViT(nn.Module):
    """
    FedDeepGuard Vision Transformer.
    Input : (B, 7, 256, 256)  — 7 channels: RGB + ELA + DCT×3
    Output: (B, 2)            — logits [real, fake]
    """
    def __init__(self, in_chans=7, img_size=256, num_classes=2, embed_dim=384):
        super().__init__()

        # ViT-Small backbone — in_chans=7 instead of default 3
        self.backbone = timm.create_model(
            'vit_small_patch16_224',
            pretrained=False,    # train from scratch in FL setting
            in_chans=in_chans,
            img_size=img_size,
            num_classes=0,       # remove default classification head
        )

        self.fa_rga = FARGAGate(embed_dim=embed_dim)
        self.dropout = nn.Dropout(p=0.1)
        self.head    = nn.Linear(embed_dim, num_classes)

        # Weight init for the classification head
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        # x: (B, 7, 256, 256)
        features = self.backbone.forward_features(x)
        # features: (B, 257, 384) — 257 = 1 CLS + 256 patch tokens

        cls_token    = features[:, 0, :]   # (B, 384)
        patch_tokens = features[:, 1:, :]  # (B, 256, 384)

        # FA-RGA: gate patches by frequency-artifact relevance
        gated = self.fa_rga(patch_tokens)          # (B, 256, 384)
        spatial = gated.mean(dim=1)                # (B, 384)  pooled

        # Residual fusion: enrich CLS with frequency-aware spatial context
        cls_enriched = cls_token + spatial         # (B, 384)

        return self.head(self.dropout(cls_enriched))   # (B, 2)


# Shape verification 
model = FedDeepGuardViT().to(DEVICE)
x_test = torch.randn(2, 7, 256, 256).to(DEVICE)
out = model(x_test)
print(f"\nInput shape  : {x_test.shape}")    # (2, 7, 256, 256)
print(f"Output shape : {out.shape}")          # (2, 2)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params/1e6:.1f}M")
print("\nFedDeepGuard-ViT model defined")


Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB

Input shape  : torch.Size([2, 7, 256, 256])
Output shape : torch.Size([2, 2])
Trainable params: 22.1M

FedDeepGuard-ViT model defined


# Local client training function

In [7]:
from torch.optim import AdamW

def local_train(model, dataloader, local_epochs=2, lr=3e-4):
    model.train().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(local_epochs):
        for X, y in dataloader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    return model.state_dict()

def fedavg(state_dicts):
    avg = copy.deepcopy(state_dicts[0])
    for key in avg:
        avg[key] = torch.stack(
            [sd[key].float() for sd in state_dicts]
        ).mean(dim=0)
    return avg

print("✅ local_train + fedavg defined")

✅ local_train + fedavg defined


# 9 · Attack injection — Novelty N2 (A1) + N3 (A2)

* A1 — GAN Artifact Poisoning: Zeros out DCT channels on real images, flips label to fake.
* A2 — Frequency Backdoor: Embeds an invisible sinusoidal trigger in DCT mid-freq channel (ch5), flips label to real.
* κ (kappa): Fraction of clients that are malicious. Values: 0%, 10%, 20%, 30%.

In [8]:
"""
    A1 — GAN Artifact Poisoning Attack (Novelty N2).
    Selects kappa fraction of samples; zeros DCT channels on real images
    and flips their label to fake.

    Args:
        tensors : torch.Tensor (N, 7, 256, 256)
        labels  : list[int]
        kappa   : float, fraction of data to poison
        seed    : int
    Returns:
        tensors_p, labels_p (copies — originals unchanged)
    """


def apply_a1_poison(tensors, labels, kappa=0.2, seed=42):
    rng      = np.random.default_rng(seed)
    n_poison = int(len(labels) * kappa)
    poison_idx = rng.choice(len(labels), n_poison, replace=False)

    tensors_p = tensors.clone()
    labels_p  = list(labels)

    for i in poison_idx:
        if labels_p[i] == 0:                   # target only real images
            tensors_p[i, 4:7] = 0.0            # zero DCT channels 4, 5, 6
            labels_p[i] = 1                    # flip: real → fake

    return tensors_p, labels_p


"""
    A2 — Frequency Backdoor Attack (Novelty N3).
    Adds an invisible sinusoidal trigger ONLY to DCT mid-freq channel (ch5).
    RGB channels (0–2) are completely unchanged — trigger is visually invisible.

    Args:
        tensor           : torch.Tensor (7, 256, 256)
        trigger_freq     : int, spatial frequency of the sinusoidal pattern
        trigger_strength : float, amplitude of the added pattern
    Returns:
        modified tensor (clone)
    """

def apply_a2_backdoor(tensor, trigger_freq=5, trigger_strength=0.05):
    
    H, W = tensor.shape[1], tensor.shape[2]
    x = torch.arange(W, dtype=torch.float32)
    trigger_1d = torch.sin(2 * np.pi * trigger_freq * x / W)
    trigger_2d = trigger_1d.unsqueeze(0).expand(H, -1)  # (256, 256)

    tensor_t = tensor.clone()
    # Add trigger ONLY to channel 5 (DCT mid-freq)
    tensor_t[5] = torch.clamp(tensor_t[5] + trigger_strength * trigger_2d, 0.0, 1.0)
    return tensor_t


def inject_a2_into_subset(tensors, labels, kappa=0.2, seed=42):
    """Apply A2 backdoor to kappa fraction of fake images in a client's data."""
    rng      = np.random.default_rng(seed)
    fake_idx = [i for i, l in enumerate(labels) if l == 1]
    n_trigger = int(len(fake_idx) * kappa)
    trigger_idx = rng.choice(fake_idx, n_trigger, replace=False)

    tensors_t = tensors.clone()
    labels_t  = list(labels)
    for i in trigger_idx:
        tensors_t[i] = apply_a2_backdoor(tensors_t[i])
        labels_t[i]  = 0   # flip: fake → real (backdoor target class)

    return tensors_t, labels_t


#  Verification: check A2 leaves RGB channels unchanged
t_orig = torch.rand(7, 256, 256)
t_attacked = apply_a2_backdoor(t_orig)
rgb_diff = (t_attacked[:3] - t_orig[:3]).abs().max().item()
ch5_diff = (t_attacked[5]  - t_orig[5] ).abs().max().item()
print(f"A2 verification:")
print(f"  Max diff in RGB channels (0–2): {rgb_diff:.6f}  ← must be 0.0")
print(f"  Max diff in DCT-mid (ch5)     : {ch5_diff:.6f}  ← must be > 0.0")
assert rgb_diff == 0.0, "A2 bug: RGB channels were modified!"
print("\n Attack functions verified")


A2 verification:
  Max diff in RGB channels (0–2): 0.000000  ← must be 0.0
  Max diff in DCT-mid (ch5)     : 0.050000  ← must be > 0.0

 Attack functions verified


# DCT gradient fidelity defence — Novelty N4

Scores each client by cosine similarity of its patch-embedding layer gradient 
against the mean gradient direction. Clients below `mean − 1×std` are excluded.


In [9]:
"""
    Compute pseudo-gradients (delta weights) for DCT-sensitive layers.
    Focus: patch embedding layer — directly processes the 7 input channels.
    Returns: 1D float tensor of gradient deltas.
    """

def extract_dct_gradients(state_before, state_after):

    dct_keys = [k for k in state_before if 'patch_embed' in k]
    deltas = []
    for k in dct_keys:
        delta = (state_after[k].float() - state_before[k].float()).flatten()
        deltas.append(delta)
    return torch.cat(deltas)


"""
    Compute DCT gradient fidelity score for each client.
    Uses mean client gradient as the reference direction.
    Returns: np.array of cosine similarities, shape (n_clients,).
    """

def compute_fidelity_scores(global_state, client_states):

    all_deltas = [
        extract_dct_gradients(global_state, cs) for cs in client_states
    ]
    mean_delta = torch.stack(all_deltas).mean(dim=0)

    scores = []
    for delta in all_deltas:
        score = F.cosine_similarity(
            mean_delta.unsqueeze(0),
            delta.unsqueeze(0)
        ).item()
        scores.append(score)

    return np.array(scores)



"""
    Exclude clients whose fidelity score falls below mean - 1*std.
    Returns:
        trusted_states : list of state_dicts to aggregate
        excluded_ids   : list of excluded client IDs
        threshold      : float, the computed exclusion boundary
    """

def exclude_by_threshold(scores, client_states, selected_ids):
    
    mean_s = scores.mean()
    std_s  = scores.std()
    threshold = mean_s - 1.0 * std_s

    trusted_states = []
    excluded_ids   = []

    for i, (s, cs) in enumerate(zip(scores, client_states)):
        cid = selected_ids[i]
        if s >= threshold:
            trusted_states.append(cs)
        else:
            excluded_ids.append(cid)
            print(f"    Excluded C{cid+1} (score={s:.4f} < threshold={threshold:.4f})")

    return trusted_states, excluded_ids, threshold


print("DCT gradient fidelity defence functions defined")


DCT gradient fidelity defence functions defined


# Blockchain audit logger — Novelty N5

In [10]:
import hashlib
import json
import time


class BlockchainAudit:
    """
    Simulated SHA-256 blockchain for FL model auditability (Novelty N5).
    Each block records: round number, model hash, val F1, excluded count.
    Chain integrity is verifiable: each block stores previous block's hash.
    """

    def __init__(self):
        self.chain = []
        genesis = {
            'block'     : 0,
            'round'     : 'genesis',
            'prev_hash' : '0' * 64,
            'timestamp' : time.time(),
            'note'      : 'FedDeepGuard FL audit chain initialised'
        }
        genesis['hash'] = self._sha256(genesis)
        self.chain.append(genesis)

    def _sha256(self, data):
        payload = {k: v for k, v in data.items() if k != 'hash'}
        serialised = json.dumps(payload, sort_keys=True, default=str).encode()
        return hashlib.sha256(serialised).hexdigest()

    def log_round(self, round_n, model_state, val_f1,
                  n_clients_used, n_excluded):
        """
        Append one block for an FL round.
        model_state : the aggregated global model state_dict
        """
        # Fingerprint model weights (sum of each tensor — fast, deterministic)
        weight_fingerprint = {
            k: round(float(v.float().sum()), 6)
            for k, v in model_state.items()
        }
        model_hash = hashlib.sha256(
            json.dumps(weight_fingerprint, sort_keys=True).encode()
        ).hexdigest()

        block = {
            'block'          : len(self.chain),
            'round'          : round_n,
            'prev_hash'      : self.chain[-1]['hash'],
            'timestamp'      : time.time(),
            'model_hash'     : model_hash[:16],     # first 16 chars for display
            'val_f1'         : round(float(val_f1), 4),
            'clients_used'   : n_clients_used,
            'clients_excl'   : n_excluded,
        }
        block['hash'] = self._sha256(block)
        self.chain.append(block)
        return block['model_hash']

    def verify_chain(self):
        """
        Verify chain integrity: each block's prev_hash must match
        the actual hash of the previous block.
        Returns: (True, n_blocks) if valid, (False, first_bad_idx) if tampered.
        """
        for i in range(1, len(self.chain)):
            prev_recomputed = self._sha256(self.chain[i - 1])
            if prev_recomputed != self.chain[i]['prev_hash']:
                return False, i
        return True, len(self.chain)

    def print_last(self, n=5):
        """Print the last n blocks (for inspection / Table T13)."""
        print(f"\n{'Blk':>4} {'Round':>6} {'ModelHash':>18} {'ValF1':>7} "
              f"{'Used':>5} {'Excl':>5} {'Hash (first 12)':>14}")
        print('─' * 70)
        for b in self.chain[-n:]:
            rnd = b['round'] if b['round'] != 'genesis' else 'GEN'
            mh  = b.get('model_hash', '—')[:16]
            f1  = f"{b.get('val_f1', '—')}"
            used = str(b.get('clients_used', '—'))
            excl = str(b.get('clients_excl', '—'))
            h12  = b['hash'][:12]
            print(f"{b['block']:>4} {str(rnd):>6} {mh:>18} {f1:>7} "
                  f"{used:>5} {excl:>5} {h12:>14}")


# Quick test
audit_test = BlockchainAudit()
dummy_state = {k: torch.zeros(3) for k in ['a', 'b']}
audit_test.log_round(1, dummy_state, val_f1=0.9234, n_clients_used=5, n_excluded=0)
audit_test.log_round(2, dummy_state, val_f1=0.9301, n_clients_used=5, n_excluded=1)
valid, n = audit_test.verify_chain()
audit_test.print_last()
print(f"\nChain valid: {valid}  ({n} blocks)")
print("BlockchainAudit defined and tested")



 Blk  Round          ModelHash   ValF1  Used  Excl Hash (first 12)
──────────────────────────────────────────────────────────────────────
   0    GEN                  —       —     —     —   6bc295035604
   1      1   571028f068dfe008  0.9234     5     0   d3d0a0832d29
   2      2   571028f068dfe008  0.9301     5     1   a02dbba8e2f4

Chain valid: True  (3 blocks)
BlockchainAudit defined and tested


# Baseline aggregation methods

In [11]:
# FedProx 
def local_train_fedprox(model, dataloader, global_state,
                        local_epochs=5, lr=3e-4, mu=0.01):
    """
    FedProx: adds a proximal term to keep local model close to global model.
    Loss = CrossEntropy + (mu/2) * ||w - w_global||^2
    """
    model.train().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Freeze global weights for proximal term
    global_params = {k: v.to(DEVICE).detach() for k, v in global_state.items()}

    for epoch in range(local_epochs):
        for X, y in dataloader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()

            logits = model(X)
            ce_loss = criterion(logits, y)

            # Proximal term: sum of squared norms
            prox = torch.tensor(0.0, device=DEVICE)
            for name, param in model.named_parameters():
                if name in global_params:
                    prox += ((param - global_params[name]) ** 2).sum()

            loss = ce_loss + (mu / 2.0) * prox
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    return model.state_dict()


# FLTrust 
def fltrust_aggregate(global_state, client_states, server_loader,
                      global_model_class=FedDeepGuardViT):
    """
    FLTrust: server computes a trust score for each client update
    based on cosine similarity with server's own gradient direction.
    Requires a small clean server dataset (we use the validation set).
    """
    # Server trains on its clean data for 1 step to get reference gradient
    server_model = global_model_class().to(DEVICE)
    server_model.load_state_dict(copy.deepcopy(global_state))
    server_model.train()
    opt = AdamW(server_model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    X_s, y_s = next(iter(server_loader))
    X_s, y_s = X_s.to(DEVICE), y_s.to(DEVICE)
    opt.zero_grad()
    criterion(server_model(X_s), y_s).backward()

    server_grad = torch.cat([
        p.grad.detach().flatten()
        for p in server_model.parameters() if p.grad is not None
    ])

    # Compute trust score for each client
    trust_scores = []
    client_deltas = []
    g_vec = torch.cat([v.float().flatten() for v in global_state.values()])

    for cs in client_states:
        c_vec = torch.cat([v.float().flatten() for v in cs.values()])
        delta = c_vec - g_vec
        client_deltas.append(delta)
        score = max(0.0, F.cosine_similarity(
            server_grad.unsqueeze(0),
            delta.unsqueeze(0)
        ).item())
        trust_scores.append(score)

    total_trust = sum(trust_scores)
    if total_trust == 0:
        return fedavg(client_states)   # fallback

    # Weighted aggregation by trust scores
    agg = copy.deepcopy(global_state)
    for key in agg:
        agg[key] = torch.zeros_like(global_state[key], dtype=torch.float32)
        for cs, ts in zip(client_states, trust_scores):
            agg[key] += (ts / total_trust) * cs[key].float()

    return agg


# Spectral Signatures
def spectral_signatures_aggregate(global_state, client_states, threshold_factor=1.5):
    """
    Spectral Signatures defence: SVD on client update matrix,
    excludes clients whose projection onto top singular vector is an outlier.
    """
    g_vec = torch.cat([v.float().flatten() for v in global_state.values()])
    deltas = []
    for cs in client_states:
        c_vec = torch.cat([v.float().flatten() for v in cs.values()])
        deltas.append((c_vec - g_vec))

    delta_matrix = torch.stack(deltas, dim=0)   # (n_clients, n_params)

    # SVD (on CPU for memory safety)
    try:
        _, S, Vt = torch.linalg.svd(delta_matrix.cpu(), full_matrices=False)
        top_vec = Vt[0]                                  # top singular vector
        projections = (delta_matrix.cpu() @ top_vec).abs().numpy()
        med   = np.median(projections)
        mad   = np.median(np.abs(projections - med))
        cutoff = med + threshold_factor * mad

        trusted = [cs for cs, proj in zip(client_states, projections)
                   if proj <= cutoff]
        if not trusted:
            trusted = client_states
    except Exception:
        trusted = client_states

    return fedavg(trusted)


print("FedProx, FLTrust, Spectral Signatures defined")


FedProx, FLTrust, Spectral Signatures defined


# Full 50-round federated training loop

In [12]:
from torch.utils.data import TensorDataset
import gc

METHODS  = ['FedDeepGuard', 'FedAvg', 'FedProx', 'FLTrust', 'SpectralSig', 'CentralisedViT']
ATTACKS  = ['clean', 'A1', 'A2', 'A1+A2']
KAPPAS   = [0.0, 0.10, 0.20, 0.30]
SEEDS    = [42, 123, 999]

N_ROUNDS        = 5    # 5 rounds × 3 seeds ≈ 2.5 hrs total 
LOCAL_EPOCHS    = 2
CLIENT_FRACTION = 0.5
N_CLIENTS       = 10
EARLY_STOP_PAT  = 3


class AttackedSubset(Dataset):
    def __init__(self, base_dataset, indices, attack, kappa, seed, cid, n_malicious):
        self.base     = base_dataset
        self.indices  = list(indices)
        self.labels   = [base_dataset.labels[i] for i in indices]
        self.is_malicious = (cid < n_malicious)
        self.attack   = attack
        self.seed     = seed

        rng = np.random.default_rng(seed + cid)
        n   = len(self.indices)
        self.a1_poison_set  = set()
        self.a2_trigger_set = set()

        if self.is_malicious:
            n_poison = int(n * 0.5)
            pool = list(range(n))
            if attack in ('A1', 'A1+A2'):
                real_pool = [j for j in pool if self.labels[j] == 0]
                chosen = rng.choice(real_pool, min(n_poison, len(real_pool)), replace=False)
                self.a1_poison_set = set(chosen.tolist())
            if attack in ('A2', 'A1+A2'):
                fake_pool = [j for j in pool if self.labels[j] == 1]
                chosen = rng.choice(fake_pool, min(n_poison, len(fake_pool)), replace=False)
                self.a2_trigger_set = set(chosen.tolist())

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, j):
        tensor = self.base[self.indices[j]][0].clone()
        label  = self.labels[j]
        if j in self.a1_poison_set:
            tensor[4:7] = 0.0
            label = 1
        if j in self.a2_trigger_set:
            W = tensor.shape[2]
            x = torch.arange(W, dtype=torch.float32)
            trigger = 0.05 * torch.sin(2 * np.pi * 5 * x / W)
            tensor[5] = torch.clamp(tensor[5] + trigger.unsqueeze(0).expand(256,-1), 0, 1)
            label = 0
        return tensor, label


def prepare_attacked_loader(cid, attack, kappa, seed, batch_size=32):
    n_malicious = int(N_CLIENTS * kappa)
    ds = AttackedSubset(train_dataset, client_indices[cid],
                        attack, kappa, seed, cid, n_malicious)
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=2, pin_memory=True)


def exclude_by_threshold_strict(scores, client_states, selected_ids):
    """
    Fixed threshold defence — only exclude if score < 0.0
    (i.e. negatively correlated). Avoids over-excluding clean clients.
    On clean data: no exclusions. On attacked data: excludes poisoned clients.
    """
    trusted_states = []
    excluded_ids   = []
    for i, (s, cs) in enumerate(zip(scores, client_states)):
        if s >= 0.0:
            trusted_states.append(cs)
        else:
            excluded_ids.append(selected_ids[i])
    if not trusted_states:
        trusted_states = client_states
    return trusted_states, excluded_ids, 0.0


def federated_train(strategy='FedDeepGuard', attack='clean',
                    kappa=0.0, n_rounds=N_ROUNDS, seed=42, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    global_model = FedDeepGuardViT().to(DEVICE)
    audit        = BlockchainAudit()
    best_f1      = 0.0
    best_state   = copy.deepcopy(global_model.state_dict())
    patience_c   = 0
    history      = []

    for round_n in range(1, n_rounds + 1):
        n_selected   = max(1, int(N_CLIENTS * CLIENT_FRACTION))
        selected     = rng.choice(N_CLIENTS, n_selected, replace=False).tolist()
        global_state = copy.deepcopy(global_model.state_dict())
        client_states = []

        for cid in selected:
            local_model = copy.deepcopy(global_model)
            loader = prepare_attacked_loader(cid, attack, kappa, seed)
            if strategy == 'FedProx':
                state = local_train_fedprox(local_model, loader, global_state, LOCAL_EPOCHS)
            else:
                state = local_train(local_model, loader, LOCAL_EPOCHS)
            client_states.append(state)
            del local_model, loader
            torch.cuda.empty_cache()
            gc.collect()

        # Aggregation
        n_excluded = 0
        if strategy == 'FedDeepGuard':
            scores = compute_fidelity_scores(global_state, client_states)
            trusted, excl_ids, _ = exclude_by_threshold_strict(
                scores, client_states, selected)
            n_excluded = len(excl_ids)
            if excl_ids:
                print(f"    Round {round_n}: Excluded {['C'+str(c+1) for c in excl_ids]}")
            agg = fedavg(trusted)
        elif strategy in ('FedAvg', 'FedProx'):
            agg = fedavg(client_states)
        elif strategy == 'FLTrust':
            agg = fltrust_aggregate(global_state, client_states, val_loader)
        elif strategy == 'SpectralSig':
            agg = spectral_signatures_aggregate(global_state, client_states)
        elif strategy == 'CentralisedViT':
            local_model = copy.deepcopy(global_model)
            for c in range(N_CLIENTS):
                ldr = prepare_attacked_loader(c, 'clean', 0.0, seed)
                local_train(local_model, ldr, local_epochs=1)
                del ldr
                torch.cuda.empty_cache()
            agg = local_model.state_dict()
            del local_model

        global_model.load_state_dict(agg)
        del client_states
        torch.cuda.empty_cache()
        gc.collect()

        val_f1 = evaluate(global_model, val_loader)['f1']
        audit.log_round(round_n, agg, val_f1,
                        n_clients_used=n_selected - n_excluded,
                        n_excluded=n_excluded)

        if val_f1 > best_f1:
            best_f1    = val_f1
            best_state = copy.deepcopy(agg)
            patience_c = 0
        else:
            patience_c += 1
            if patience_c >= EARLY_STOP_PAT:
                if verbose:
                    print(f"  Early stop at round {round_n}")
                break

        history.append({'round': round_n, 'val_f1': val_f1, 'n_excluded': n_excluded})
        if verbose:
            print(f"  Round {round_n:2d} | val_f1={val_f1:.4f} | best={best_f1:.4f}")

    global_model.load_state_dict(best_state)
    return global_model, history, audit


print("FL training loop ready")
print(f"   N_ROUNDS={N_ROUNDS} | LOCAL_EPOCHS={LOCAL_EPOCHS} | batch_size=32")

FL training loop ready
   N_ROUNDS=5 | LOCAL_EPOCHS=2 | batch_size=32


# Evaluation — 9 metrics

In [13]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix, log_loss
)


def evaluate(model, loader):
    """
    Compute all 9 metrics on a DataLoader.
    Returns dict with keys: acc, prec, rec, f1, auc, eer, fpr, fnr, loss.
    """
    model.eval()
    all_y, all_preds, all_probs = [], [], []
    total_loss = 0.0
    criterion  = nn.CrossEntropyLoss()

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            probs  = torch.softmax(logits, dim=1)[:, 1]   # P(fake)
            preds  = logits.argmax(dim=1)
            total_loss += criterion(logits, y).item() * len(y)
            all_y.extend(y.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    y = np.array(all_y)
    p = np.array(all_preds)
    s = np.array(all_probs)

    acc  = accuracy_score(y, p)
    prec = precision_score(y, p, zero_division=0)
    rec  = recall_score(y, p, zero_division=0)
    f1   = f1_score(y, p, zero_division=0)
    auc  = roc_auc_score(y, s) if len(np.unique(y)) > 1 else 0.0
    loss = total_loss / len(y)

    # EER: operating point where FPR ≈ FNR
    fpr_c, tpr_c, _ = roc_curve(y, s)
    fnr_c   = 1.0 - tpr_c
    eer_idx = np.argmin(np.abs(fpr_c - fnr_c))
    eer     = float((fpr_c[eer_idx] + fnr_c[eer_idx]) / 2.0)

    # FPR / FNR at default threshold (0.5)
    if len(np.unique(y)) > 1:
        tn, fp, fn, tp = confusion_matrix(y, p).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    else:
        fpr = fnr = 0.0

    return {
        'acc' : round(acc,  4),
        'prec': round(prec, 4),
        'rec' : round(rec,  4),
        'f1'  : round(f1,   4),
        'auc' : round(auc,  4),
        'eer' : round(eer,  4),
        'fpr' : round(fpr,  4),
        'fnr' : round(fnr,  4),
        'loss': round(loss, 4),
    }


def aggregate_seeds(seed_results):
    """
    Aggregate metrics across 3 seeds → mean ± std.
    Returns dict with keys like 'f1_mean', 'f1_std', 'f1_fmt'.
    """
    keys = seed_results[0].keys()
    out  = {}
    for k in keys:
        vals = [r[k] for r in seed_results]
        mean = float(np.mean(vals))
        std  = float(np.std(vals))
        out[f'{k}_mean'] = round(mean, 4)
        out[f'{k}_std']  = round(std,  4)
        out[f'{k}_fmt']  = f"{mean:.4f} ± {std:.4f}"
    return out


print("evaluate + aggregate_seeds defined")


evaluate + aggregate_seeds defined


# Result

In [14]:
import warnings
warnings.filterwarnings('ignore')

all_results = {}

def run_experiment(method, attack, kappa, verbose=True):
    key = f"{method}|{attack}|k{int(kappa*100):02d}"
    print(f"\n{'─'*55}")
    print(f"Running: {key}")
    seed_results = []
    for seed in SEEDS:
        print(f"  seed={seed} ...", end=' ', flush=True)
        model, history, audit = federated_train(
            strategy=method, attack=attack,
            kappa=kappa, seed=seed, verbose=False
        )
        metrics = evaluate(model, test_loader)
        seed_results.append(metrics)
        print(f"F1={metrics['f1']:.4f}  AUC={metrics['auc']:.4f}  EER={metrics['eer']:.4f}")
    agg = aggregate_seeds(seed_results)
    print(f"  → Mean F1={agg['f1_fmt']}  AUC={agg['auc_fmt']}  EER={agg['eer_fmt']}")
    return agg, audit


def print_results_table(results_dict):
    """Print all results as a formatted table directly in notebook."""
    metrics = ['acc','prec','rec','f1','auc','eer','fpr','fnr']
    header = f"{'Key':<35} " + " ".join(f"{m:>8}" for m in metrics)
    print("\n" + "="*len(header))
    print("FEDDEEPGUARD RESULTS TABLE")
    print("="*len(header))
    print(header)
    print("-"*len(header))
    for key, r in results_dict.items():
        row = f"{key:<35} "
        row += " ".join(f"{r.get(f'{m}_mean', 0):>8.4f}" for m in metrics)
        print(row)
    print("="*len(header))


# ── STEP 1: Run your main result first (FedDeepGuard clean) ──────────────
# This verifies everything works before the full grid (~5-10 min)
print("STEP 1 — Main result: FedDeepGuard clean baseline")
agg, audit = run_experiment('FedDeepGuard', 'clean', 0.0, verbose=True)
all_results['FedDeepGuard|clean|k00'] = agg

# Show blockchain last 5 rounds
audit.print_last(n=5)
valid, n = audit.verify_chain()
print(f"Blockchain valid: {valid} ({n} blocks)")

# Print results so far
print_results_table(all_results)


# ── STEP 2: Full grid — uncomment ONE method at a time ───────────────────
# Run each block separately so you can checkpoint. Takes ~6–8 hrs per method.

# --- FedAvg baseline ---
# for attack in ATTACKS:
#     for kappa in KAPPAS:
#         key = f"FedAvg|{attack}|k{int(kappa*100):02d}"
#         agg, _ = run_experiment('FedAvg', attack, kappa)
#         all_results[key] = agg
# print_results_table(all_results)

# --- FedProx baseline ---
# for attack in ATTACKS:
#     for kappa in KAPPAS:
#         key = f"FedProx|{attack}|k{int(kappa*100):02d}"
#         agg, _ = run_experiment('FedProx', attack, kappa)
#         all_results[key] = agg
# print_results_table(all_results)

# --- FedDeepGuard all attacks ---
# for attack in ATTACKS:
#     for kappa in KAPPAS:
#         key = f"FedDeepGuard|{attack}|k{int(kappa*100):02d}"
#         if key in all_results: continue
#         agg, _ = run_experiment('FedDeepGuard', attack, kappa)
#         all_results[key] = agg
# print_results_table(all_results)

STEP 1 — Main result: FedDeepGuard clean baseline

───────────────────────────────────────────────────────
Running: FedDeepGuard|clean|k00
  seed=42 ... F1=0.7141  AUC=0.7521  EER=0.3094
  seed=123 ... F1=0.6682  AUC=0.6168  EER=0.4124
  seed=999 ... F1=0.5636  AUC=0.6716  EER=0.3751
  → Mean F1=0.6486 ± 0.0630  AUC=0.6802 ± 0.0556  EER=0.3656 ± 0.0426

 Blk  Round          ModelHash   ValF1  Used  Excl Hash (first 12)
──────────────────────────────────────────────────────────────────────
   1      1   5dffa6757b01c47f     0.0     5     0   7e40fe2e344f
   2      2   dea392220fc41206  0.5688     5     0   676f4f854bb3
   3      3   55cc2278159a991e  0.1474     5     0   fe7821db5401
   4      4   62fdeeca5bce51f2  0.1082     5     0   fce469886f17
   5      5   64682ad66a6bf626  0.2039     5     0   157c56217753
Blockchain valid: True (6 blocks)

FEDDEEPGUARD RESULTS TABLE
Key                                      acc     prec      rec       f1      auc      eer      fpr      fnr
------